# 01. Comprehensions

**Tier:** Intermediate. **Why you'd use this:** You have a list of prices and need a new list of those prices with tax, skipping any that are zero. A short loop would work. A comprehension writes the same idea in one expression.

**After this lesson you can:** build a filtered list in one line; read a comprehension left to right; explain why the loop name inside it does not overwrite a name outside.
About 25 minutes. **Uses:** [Loops](../03-control-flow/02-loops-lesson.ipynb), [Lists](../04-collections/01-lists-lesson.ipynb)


## Quick review

1. Why does `finally` run after a `return` inside `try`?
2. What does `list(grid)` share that `deepcopy` does not?

<details><summary>Answers</summary>

Cleanup must not be skipped. A shallow copy shares nested rows.
</details>


## The idea

A list comprehension builds a new list from a loop written inside brackets. Read it as "collect this expression for each item, optionally only when a filter is true." The loop name lives only inside that expression. It does not change a name you already had outside.

<details><summary>Explain more</summary>

`[price * 1.1 for price in prices if price > 0]` is "for each price, if it is positive, keep price times 1.1." The result is a fresh list. The input list is unchanged.
</details>


## See it work

Tax the positive prices and leave the zeros out.


In [1]:
def taxed(prices):
    # 1. One expression: value, source, filter.
    return [price * 1.1 for price in prices if price > 0]


print(taxed([10, 0, 5]))


[11.0, 5.5]


## Check yourself

Answer these before you run the cell.

1. What does the comprehension print?
2. After it runs, what is `item` still bound to?


In [2]:
item = "keep"
print([item for item in ("a", "b")])
print(item)


['a', 'b']
keep


<details><summary>Answers</summary>

The list is `['a', 'b']`. `item` is still `"keep"`. The name inside the comprehension is a different, local name.
</details>


## Why it works this way

A comprehension is compiled as a tiny nested scope, like a short function that runs immediately and returns a list. The loop name is local to that scope. Assigning `item` inside the brackets never writes the outer `item`. That is why the outer name survives after the brackets finish. In a plain `for` loop the same name would be reused and left bound to the last value. The comprehension rule exists so a short expression does not quietly rewrite names in the surrounding function. Deep dive: [Scopes and closures](../../internals/06-scopes/01-closures-lesson.ipynb).

The comprehension still walks every source item once. Filtering with `if` skips appends, but it does not skip the walk. Time is proportional to the length of the input, not only to how many values you keep. Memory is one new list that holds the kept results. A plain loop that appends does the same work with more lines. The design is readability for map-and-filter shapes, not a different algorithm with a better big-O cost.

Read left to right: output expression, then `for`, then optional `if`. Nesting a second `for` is legal. It is often harder to read than two steps, especially when each step needs a name. Prefer a clear loop when the expression grows past one idea, when you need `break`, or when you need to update more than one collection. The brackets are a contract with the reader: this expression builds one list and has no other job.

The own-scope rule also matters for late binding surprises later. A comprehension that captures a loop name into a nested function still sees that name through the comprehension's own locals while the expression runs. Treat the brackets as a mini function body. If you need the last walked value outside, write a loop and assign on purpose. Relying on a leaked name is the old behaviour and is gone on purpose.


In [3]:
item = "keep"
built = [item for item in ("a", "b")]
print(built)
print(item)
print([n * n for n in range(4) if n % 2 == 0])


['a', 'b']
keep
[0, 4]


## When to use it

Use a comprehension for a new list that is a map or filter of another. Use a loop when you need several statements, early `break`, or a side effect. Don't pack a comprehension with nested loops and two filters. Clarity beats one-liners.


## Common mistake

Expecting the loop name to leak out, then using it after the comprehension. The outer name never changed, so later code sees the old value and looks "wrong" in silence.


In [4]:
item = "keep"
[item for item in ("a", "b")]
print(item)


keep


If you need the last value from a walk, write a loop and assign on purpose. Do not rely on a comprehension to leave a name behind.


In [5]:
item = "keep"
for item in ("a", "b"):
    pass
print(item)


b


## Try it

Change `taxed` so prices of exactly `0` stay in the result as `0.0`, and only negative prices are dropped. Which part of the comprehension changes?


## Recap

- A comprehension builds a new list from an expression, a `for`, and an optional `if`.
- The loop name is local to the comprehension.
- Prefer a loop when the body needs more than one expression.

**Next:** [practice](01-comprehensions-practice.ipynb)
